In [ ]:
#@title Cell F2.1 — Figure 2 design and interpretation
from IPython.display import display, Markdown
display(Markdown(r"""
# Figure 2 — Antibiotic-specific predictive error intervals (BMC-N)

**Final agreed layout**

- All **26 antibiotics**, with **one light-grey rectangular interval per row**, touching vertically.
- The plotting area has **26 rows × 4 mm = 104 mm** total height; bars have thin black borders.
- Both **top and bottom x-axes** show the same tick values, with light dashed vertical guides. The zero line is slightly stronger.
- Horizontal axis: **observed − Model C predicted log₂ MIC**. Negative values indicate overprediction; positive values indicate underprediction.
- For each antibiotic, the lower and upper ends are the **medians of its five fold-specific BMC-N 95% predictive error limits**. This summarises fitted ranges, and is **not a newly calibrated 95% interval**.
- Intervals are drawn at their **actual asymmetric error limits**, not artificially centred at zero. No endpoints are truncated.
- Model C's original overall point-prediction MAE was **0.5931 log₂ units**; the broader 95% ranges describe uncertainty for individual predictions, not typical point error.

Only the **existing NB51 held-out interval output** is read. No model fitting or new prediction is performed.
"""))

In [ ]:
#@title Cell F2.2 — Locate final NB51 output anywhere under Google Drive
from pathlib import Path
import os, pandas as pd, numpy as np, matplotlib.pyplot as plt
try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
except ImportError:
    pass
ROOT=Path(os.environ.get('BMC_PROJECT_ROOT','/content/drive/MyDrive'))
FILENAME='51_All_Four_Methods_Independent_Heldout_Intervals.csv.gz'
found=sorted(ROOT.rglob(FILENAME)) if ROOT.is_dir() else []
if not found:
    raise FileNotFoundError(f'Missing {FILENAME} under {ROOT}. Run NB51 Cell 51.9 first, or set BMC_PROJECT_ROOT.')
# Prefer established results tree if multiple copies are present.
found.sort(key=lambda p: ('RefMIC/03_Results/51_BMC_Final_Comparison' not in str(p),len(str(p))))
PATH=found[0]
OUT=PATH.parent
print('Reading:',PATH)
D=pd.read_csv(PATH,compression='gzip')
needed={'antibiotic','method','outer_fold','lower95_log2','upper95_log2','model_c_outer_test_predicted_log2_mic','interaction_row'}
missing=needed-set(D.columns)
if missing: raise ValueError(f'NB51 interval output missing columns: {missing}')
if len(D)!=4*50460: raise ValueError(f'Expected 201,840 four-method interval records, got {len(D)}')
methods=['BMC-N','BMC-T','BMC-AT','Empirical']
if sorted(D.method.unique().tolist())!=sorted(methods): raise ValueError('Unexpected method labels')
if D.antibiotic.nunique()!=26 or D.outer_fold.nunique()!=5: raise ValueError('Expected 26 antibiotics across 5 outer folds')
for side in ['lower95','upper95']:
    D['err_'+side]=D[side+'_log2']-D['model_c_outer_test_predicted_log2_mic']
if not (D.err_lower95<=D.err_upper95).all():raise ValueError('Error lower limit exceeds upper limit')
# Per antibiotic × method × fold, a common predictive error interval is expected.
within=D.groupby(['method','antibiotic','outer_fold'],observed=True).agg(
    err_lo=('err_lower95','median'), err_hi=('err_upper95','median'),
    lo_spread=('err_lower95',lambda s:s.max()-s.min()),
    hi_spread=('err_upper95',lambda s:s.max()-s.min()),
    n_test=('interaction_row','size')).reset_index()
if (within[['lo_spread','hi_spread']].to_numpy()>1e-5).any():
    raise ValueError('Unexpected within-fold variation in predictive error limits; investigate before plotting')
if len(within)!=4*26*5:raise ValueError(f'Expected 520 fold-method-antibiotic records; found {len(within)}')
summary=within.groupby(['method','antibiotic'],observed=True).agg(
    lower=('err_lo','median'),upper=('err_hi','median'),
    min_fold_lower=('err_lo','min'),max_fold_upper=('err_hi','max'),
    n_folds=('outer_fold','nunique')).reset_index()
assert summary.n_folds.eq(5).all() and len(summary)==104
summary['width']=summary.upper-summary.lower
print('Validated 104 method × antibiotic summaries, using all 5 folds.')
print('Plot will use only BMC-N; other methods remain available in the input for checks.')
display(summary[summary.antibiotic.isin(['ceftazidime','cefepime','ciprofloxacin','meropenem'])].round(3).head(16))

In [ ]:
#@title Cell F2.3 — Draw Figure 2: touching 4 mm rectangles, dual x-axes, black-and-white
from matplotlib.patches import Rectangle
from matplotlib.ticker import MultipleLocator

# Single BMC-N predictive error range per antibiotic; widest at the top.
plot_data=(summary.loc[summary['method'].eq('BMC-N')]
           .sort_values(['width','antibiotic'],ascending=[False,True])
           .reset_index(drop=True))
if len(plot_data)!=26 or plot_data.antibiotic.nunique()!=26 or not plot_data.n_folds.eq(5).all():
    raise ValueError('Expected 26 antibiotics with five validated BMC-N folds each.')
if not (plot_data.lower < plot_data.upper).all():
    raise ValueError('Nonpositive interval width found.')

# Exact physical bar height: the axes is 26 × 4 mm = 104 mm tall.
mm_to_in=1/25.4
bar_mm=4.0
axes_h_in=26*bar_mm*mm_to_in
fig_w_in=170*mm_to_in          # nominal full-column/two-column page width: 17 cm
fig_h_in=132*mm_to_in          # total page height: 13.2 cm including margins
axes_bottom=17*mm_to_in/fig_h_in
axes_height=axes_h_in/fig_h_in
axes_left=0.355             # sufficient room for long antibiotic labels
axes_right=0.975
if axes_bottom + axes_height >= 0.96:
    raise ValueError('Insufficient headroom for upper x-axis.')

fig=plt.figure(figsize=(fig_w_in,fig_h_in))
ax=fig.add_axes([axes_left,axes_bottom,axes_right-axes_left,axes_height])
y=np.arange(26)

# Ranges: always include zero and both genuine ends, with a small safety margin.
lo=float(min(0,plot_data.lower.min()))
hi=float(max(0,plot_data.upper.max()))
span=hi-lo
margin=max(0.16,0.025*span)
ax.set_xlim(lo-margin,hi+margin)
ax.set_ylim(25.5,-0.5)

# Dashed major vertical guides run through the plot behind the bars.
ax.set_axisbelow(True)
ax.grid(axis='x',which='major',linestyle=(0,(2.4,3.0)),lw=0.42,color='0.72',alpha=0.7,zorder=0)

# One bar per row, each is 1 data unit = 4 mm high; no spaces between rows.
for idx,r in plot_data.iterrows():
    bar=Rectangle((float(r.lower),idx-0.5),float(r.upper-r.lower),1.0,
                  facecolor='0.83',edgecolor='0.20',linewidth=0.34,
                  joinstyle='miter',zorder=2)
    ax.add_patch(bar)

# More visible zero reference line; keeps the same scale on both axes.
ax.axvline(0,color='0.35',linewidth=0.78,zorder=3)
ax.set_yticks(y)
ax.set_yticklabels(plot_data.antibiotic,fontsize=7.5)
ax.tick_params(axis='y',length=0,pad=6)
ax.tick_params(axis='x',labelsize=8,length=3.2,width=0.55,pad=3)
ax.set_xlabel('Prediction error (observed − predicted log₂ MIC)',fontsize=8.6,labelpad=7)

# Mirror tick labels at the top, with no second xlabel.
ax.tick_params(axis='x',top=True,labeltop=True,bottom=True,labelbottom=True)

# Plain borders and restrained styling; no title inside the panel.
for name in ('left','right'):
    ax.spines[name].set_visible(False)
for name in ('top','bottom'):
    ax.spines[name].set_linewidth(0.58)
    ax.spines[name].set_color('0.20')

# Verify physical height of each rectangle in the exported PDF/PNG figure.
assert abs(axes_h_in/26*25.4-4.0)<1e-9

# Use paper-safe output names, not the earlier multi-method filename.
png=OUT/'Figure02_BMCN_26_Antibiotic_Predictive_Intervals_v0.3.png'
pdf=OUT/'Figure02_BMCN_26_Antibiotic_Predictive_Intervals_v0.3.pdf'
csv=OUT/'Figure02_BMCN_26_Antibiotic_Predictive_Limits_v0.3.csv'
fig.savefig(png,dpi=350,facecolor='white')
fig.savefig(pdf,facecolor='white')
plot_data[['antibiotic','n_folds','lower','upper','width']].to_csv(csv,index=False)
plt.show();plt.close(fig)
print('Saved PNG:',png)
print('Saved PDF:',pdf)
print('Saved BMC-N interval source data:',csv)
print(f'Figure dimensions: {170:.0f} × {132:.0f} mm; bars: 4.0 mm high, no gaps.')
print('All 26 ranges use the medians of the actual five outer-fold error interval limits.')

In [ ]:
#@title Cell F2.4 — Final audit of Figure 2 endpoints and source
# All displayed endpoints come directly from validated NB51 fold medians.
assert set(plot_data.antibiotic)==set(summary.loc[summary.method.eq('BMC-N'),'antibiotic'])
assert plot_data[['lower','upper','width']].notna().all().all()
assert (plot_data['width']>0).all()
print('NB51 input:', PATH)
print('Antibiotics:',len(plot_data),' | source folds per antibiotic:',sorted(plot_data.n_folds.unique()))
print('Smallest and largest interval width (log₂):',
      round(plot_data.width.min(),4),round(plot_data.width.max(),4))
print('Figure uses no artificial centring and does not truncate actual endpoints.')
display(plot_data[['antibiotic','lower','upper','width']].round(3))